In [0]:
df_customers = spark.read.table('my_ecommerce_store.bronze.olist_customers')
df_customers.show(1)
df_orders = spark.read.table('my_ecommerce_store.bronze.olist_orders')
df_orders.show(1)
df_products = spark.read.table('my_ecommerce_store.bronze.olist_products')
df_products.show(1)
df_payments = spark.read.table('my_ecommerce_store.bronze.olist_order_payments')
df_payments.show(1)
df_order_items = spark.read.table('my_ecommerce_store.bronze.olist_order_items')
df_order_items.show(1)


+--------------------+--------------------+------------------------+-------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+--------------------+--------------------+------------------------+-------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|       franca|            SP|
+--------------------+--------------------+------------------------+-------------+--------------+
only showing top 1 row
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----

In [0]:
df_customers.dropna().drop_duplicates(['customer_id'])
df_orders.dropna().drop_duplicates(['order_id'])        
df_products.dropna().drop_duplicates(['product_id'])        
df_payments.dropna().drop_duplicates(['order_id'])        
df_order_items.dropna().drop_duplicates(['order_id'])

DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]

In [0]:
from pyspark.sql.functions import lit, concat

df_customers = df_customers.withColumn(
    'customer_address',
    concat(
        df_customers['customer_city'],
        lit(', '),
        df_customers['customer_state'],
        lit(' - '),
        df_customers['customer_zip_code_prefix']
    )
)
df_customers = df_customers.drop('customer_city', 'customer_state', 'customer_zip_code_prefix')
#df_customers.show(1)


In [0]:
df_customers.write.mode('overwrite').saveAsTable('my_ecommerce_store.silver.olist_customers')


In [0]:
from pyspark.sql.functions import col, sum

null_count = df_orders.select([sum(col(c).isNull().cast('int')).alias(c) for c in df_orders.columns])

null_cols_with_count = {
         col_name: count for col_name, count in null_count.first().asDict().items() if count > 0
}

for col_name, count in null_cols_with_count.items():
  print(f'column {col_name} has {count} null values.')

In [0]:
silver_orders = df_orders \
        .join(df_customers, df_orders['customer_id'] == df_customers['customer_id'], 'left' ) \
        .join(df_order_items, df_orders['order_id'] == df_order_items['order_id'], 'left' ) \
        .join(df_products, df_order_items['product_id'] == df_products['product_id'], 'left' ) \
        .join(df_payments, df_orders['order_id'] == df_payments['order_id'], 'left' ) 
        

In [0]:
silver_orders_final = silver_orders.select(
    df_orders['order_id'],
    df_orders['customer_id'],
    df_orders['order_status'],
    df_orders['order_purchase_timestamp'],
    df_products['product_category_name'],
    df_order_items['price'],
    df_payments['payment_value']
)

display(silver_orders_final)


order_id,customer_id,order_status,order_purchase_timestamp,product_category_name,price,payment_value
f373335aac9a659de916f7170b8bc07a,f06a94a401e52fb019c72f2e8bbf6a2f,shipped,2018-03-17T15:32:31Z,bebes,35.9,59.18
118045506e1c1dda060171af43fe11b4,638c6674418fc58283a73c078bcb076f,delivered,2018-03-08T19:06:05Z,moveis_decoracao,45.0,360.4
118045506e1c1dda060171af43fe11b4,638c6674418fc58283a73c078bcb076f,delivered,2018-03-08T19:06:05Z,moveis_decoracao,45.0,360.4
118045506e1c1dda060171af43fe11b4,638c6674418fc58283a73c078bcb076f,delivered,2018-03-08T19:06:05Z,moveis_decoracao,45.0,360.4
118045506e1c1dda060171af43fe11b4,638c6674418fc58283a73c078bcb076f,delivered,2018-03-08T19:06:05Z,moveis_decoracao,45.0,360.4
118045506e1c1dda060171af43fe11b4,638c6674418fc58283a73c078bcb076f,delivered,2018-03-08T19:06:05Z,moveis_decoracao,45.0,360.4
cc66dee6fbc18bb79903c3a2cc14ff52,19d3b3a2d4756af17603e2c35c7c2815,delivered,2018-04-12T14:37:29Z,perfumaria,117.7,136.4
f44cb69655f8e4d13e7aae7cdd3d3c25,eab62436056c6ce3853a17dd6892951a,delivered,2018-07-13T22:22:57Z,relogios_presentes,155.97,18.0
f44cb69655f8e4d13e7aae7cdd3d3c25,eab62436056c6ce3853a17dd6892951a,delivered,2018-07-13T22:22:57Z,relogios_presentes,155.97,146.32
edcc6b79e8394346ba3ba21b00b4055e,08aea10c40f606e52597486db2b56a81,delivered,2018-04-29T16:03:47Z,pet_shop,99.9,162.63


In [0]:
silver_orders = silver_orders_final

In [0]:
silver_orders.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('my_ecommerce_store.silver.silver_orders')

